In [ ]:
#%pip install -U sentence-transformers datasets accelerate

In [ ]:
#%pip install pandas

In [ ]:
#%pip install torch --index-url https://download.pytorch.org/whl/cu126

In [ ]:
#%pip install datasets

In [ ]:
import pandas as pd

restaurant = pd.read_parquet("restaurant_global.parquet")
review = pd.read_parquet("review_global.parquet")

print("Restaurants:", restaurant.shape)
print("Reviews:", review.shape)

restaurant_info = restaurant[["business_id", "name", "categories", "city", "stars"]].copy()

df = review.merge(restaurant_info,on="business_id",how="inner",validate="many_to_one")

df.head()

# join the two parquets. 

In [ ]:
df["categories"]

In [ ]:
training_df = df.sample(n=min(100_000, len(df)),random_state=42).copy()

training_df["anchor"] = ("Restaurant categories: " + training_df["categories"])
training_df["positive"] = training_df["text"]
training_df = training_df[["business_id", "anchor", "positive"]].drop_duplicates(subset=["anchor", "positive"])

print("Training pairs:", len(training_df))

train_dataset = training_df.reset_index(drop=True)

print(train_dataset.head())

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [ ]:
from datasets import Dataset
pair_ls = [(pos, anch) for pos, anch in zip(training_df["positive"], training_df["anchor"])]

trn = pair_ls[:80000]
tst = pair_ls[80000:100000]
#t2 = pair_ls[110000:120000]

train_dataset = Dataset.from_dict({"anchor": [pos for pos, anch in trn],"positive": [anch for pos, anch in trn]})

test_dataset = Dataset.from_dict({"anchor": [pos for pos, anch in tst],"positive": [anch for pos, anch in tst]})
#test2_dataset = Dataset.from_dict({"anchor": [pos for pos, anch in t2],"positive": [anch for pos, anch in t2]})

In [ ]:
len(tst)

In [ ]:
# import torch

# print("CUDA:", torch.cuda.is_available())
# print("GPU count:", torch.cuda.device_count())

# if torch.cuda.is_available():
#     print(torch.cuda.get_device_name(0))
#     print(torch.cuda.get_device_capability(0))

In [ ]:
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)
from sentence_transformers.losses import MultipleNegativesRankingLoss


model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

model.max_seq_length = 128

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="restaurant-model-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    save_strategy="steps",
    save_steps=250,
    logging_steps=25,
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss
)

trainer.train()

model.save_pretrained("restaurant-model-finetuned")

print("Fine-tuning complete!")

In [ ]:
from sentence_transformers import util

model = SentenceTransformer("restaurant-model-finetuned")

categories = "Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries"
review = "The bubble tea was delicious and the pastries were fresh."

embeddings = model.encode([categories, review],convert_to_tensor=True,normalize_embeddings=True)

similarity = util.cos_sim(embeddings[0], embeddings[1]).item()

print(f"Cosine similarity: {similarity:.4f}")

In [ ]:
import numpy as np
import torch
from sentence_transformers import util

anchors = test_dataset["anchor"]
positives = test_dataset["positive"]

n = min(1000, len(anchors))
anchors = anchors[:n]
positives = positives[:n]

anchor_embeddings = model.encode(
    anchors,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

positive_embeddings = model.encode(
    positives,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

scores = util.cos_sim(anchor_embeddings, positive_embeddings)

ranked_indices = scores.argsort(dim=1, descending=True)

# Fix: create target indices on the same device
targets = torch.arange(n, device=scores.device).unsqueeze(1)

correct_positions = (
    ranked_indices == targets
).nonzero(as_tuple=True)[1]

top1 = (correct_positions == 0).float().mean().item()
top5 = (correct_positions < min(5, n)).float().mean().item()
top10 = (correct_positions < min(10, n)).float().mean().item()

print(f"Test examples: {n}")
print(f"Top-1 accuracy: {top1:.2%}")
print(f"Recall@5:       {top5:.2%}")
print(f"Recall@10:      {top10:.2%}")